# Exp 027b: HGNetV2 Soundscape Distill From exp_015d

Train a native HGNetV2 student on fully labeled soundscape rows using `exp_015d` as a fixed teacher.


## Plan

1. Load the fold-aware teacher cache from `exp_027a`.
2. Train a soundscape-only HGNetV2 student initialized from the matching `exp_011` fold checkpoint.
3. Combine supervised BCE with teacher-target distillation and export OOF-style validation outputs for later blend checks.


In [13]:
from __future__ import annotations

import gc
import json
import os
import random
import typing as tp
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj: object) -> None:
        print(obj)


In [14]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'PROJECT_STATE.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not resolve repository root')


ROOT = find_repo_root()
DATA = ROOT / 'data' / 'birdclef-2026'
EXPERIMENTS = ROOT / 'experiments'
EXP011_DIR = EXPERIMENTS / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised'
TEACHER_CACHE_DIR = None         # optional explicit path to completed exp_027a outputs


@dataclass
class Config:
    experiment_id: str = 'exp_027b'
    experiment_name: str = 'hgnetv2_soundscape_distill_from_exp015d'
    fold: int = 0
    n_folds: int = 4
    random_seed: int = 1098

    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)

    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    pretrained: bool = True
    drop_path_rate: float = 0.0

    epochs: int = 8
    warmup_epochs: int = 2
    batch_size: int = 16
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    num_workers: int = 0
    use_amp: bool = True
    distill_weight: float = 0.35
    distill_temperature: float = 1.0
    teacher_weight_power: float = 1.0
    min_teacher_confidence_train: float = 0.0

    max_train_rows: int | None = None
    max_valid_rows: int | None = None

    selection_metric: str = 'soundscape_macro_auc'
    save_every_epoch: bool = True


CFG = Config()
RUN_TRAINING = True
RESUME_TRAINING = True


def has_teacher_cache(path: Path) -> bool:
    return (path / 'teacher_meta.parquet').exists() and (path / 'teacher_outputs.npz').exists()


def resolve_teacher_dir(explicit: str | None = None) -> Path:
    candidates: list[Path] = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    candidates.extend([
        EXPERIMENTS / 'outputs' / 'exp_027a_exp015d_teacher_cache',
        EXPERIMENTS / 'outputs' / 'exp_027a_exp015d_teacher_cache' / f'fold_{CFG.fold:02d}',
        ROOT / 'outputs' / 'exp_027a_exp015d_teacher_cache',
        Path.home() / 'Downloads' / 'exp_027a_exp015d_teacher_cache',
        Path.home() / 'Downloads' / 'exp_027a_exp015d_teacher_cache' / f'fold_{CFG.fold:02d}',
    ])
    for candidate in candidates:
        if has_teacher_cache(candidate):
            return candidate

    search_roots = [
        EXPERIMENTS / 'outputs',
        Path.home() / 'Downloads',
    ]
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for meta_path in search_root.rglob('teacher_meta.parquet'):
            parent = meta_path.parent
            if has_teacher_cache(parent) and 'exp_027a' in str(parent):
                return parent

    raise FileNotFoundError(
        'Could not resolve exp_027a teacher cache. '
        'Run exp_027a first and make sure it writes teacher_meta.parquet + teacher_outputs.npz, '
        'or set TEACHER_CACHE_DIR to that completed output folder.'
    )


TEACHER_DIR = resolve_teacher_dir(TEACHER_CACHE_DIR)

OUTPUT_DIR = EXPERIMENTS / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}' / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [15]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def autocast_context() -> tp.ContextManager[object]:
    if amp_enabled:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


set_random_seed(CFG.random_seed)
device = pick_device()
amp_enabled = bool(CFG.use_amp and device.type == 'cuda')
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print({
    'root': str(ROOT),
    'teacher_dir': str(TEACHER_DIR),
    'exp011_dir': str(EXP011_DIR),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'output_dir': str(OUTPUT_DIR),
})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'teacher_dir': '/Users/yaroslav/Downloads/exp_027a_exp015d_teacher_cache', 'exp011_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_011_hgnetv2_soundscape_supervised', 'device': 'mps', 'amp_enabled': False, 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_027b_hgnetv2_soundscape_distill_from_exp015d/fold_00'}


In [16]:
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
CLASSES = taxonomy['primary_label'].astype(str).tolist()
N_CLASSES = len(CLASSES)
label2idx = {label: idx for idx, label in enumerate(CLASSES)}

def resolve_soundscape_path(filename: str, *raw_paths: object) -> str:
    candidates: list[Path] = []
    for raw in raw_paths:
        if raw is None:
            continue
        raw_str = str(raw)
        if raw_str in {'', 'None', 'nan', '<NA>'}:
            continue
        path = Path(raw_str).expanduser()
        candidates.append(path)
        candidates.append(Path(raw_str.replace('/kaggle/input/competitions/birdclef-2026', str(DATA))))
        candidates.append(Path(raw_str.replace('/kaggle/input/birdclef-2026', str(DATA))))
    candidates.append(DATA / 'train_soundscapes' / filename)
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(DATA / 'train_soundscapes' / filename)


teacher_meta = pd.read_parquet(TEACHER_DIR / 'teacher_meta.parquet')
teacher_arr = np.load(TEACHER_DIR / 'teacher_outputs.npz')
teacher_logits = teacher_arr['teacher_logits'].astype(np.float32)
teacher_probs = teacher_arr['teacher_probs'].astype(np.float32)
teacher_labels = teacher_arr['labels'].astype(np.float32)

assert len(teacher_meta) == teacher_logits.shape[0] == teacher_probs.shape[0] == teacher_labels.shape[0]
if 'fold' not in teacher_meta.columns:
    raise KeyError('Teacher cache is missing fold assignments')

teacher_meta = teacher_meta.copy()
source_file_values = teacher_meta['source_file_path'].tolist() if 'source_file_path' in teacher_meta.columns else [None] * len(teacher_meta)
teacher_meta['file_path'] = [
    resolve_soundscape_path(filename, file_path, source_file_path)
    for filename, file_path, source_file_path in zip(
        teacher_meta['filename'].tolist(),
        teacher_meta['file_path'].tolist(),
        source_file_values,
    )
]
if 'source_file_path' in teacher_meta.columns:
    teacher_meta['source_file_path'] = [
        resolve_soundscape_path(filename, source_file_path, file_path)
        for filename, source_file_path, file_path in zip(
            teacher_meta['filename'].tolist(),
            source_file_values,
            teacher_meta['file_path'].tolist(),
        )
    ]

missing_paths = [path for path in teacher_meta['file_path'].tolist() if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(
        f'Failed to resolve {len(missing_paths)} teacher soundscape paths. '
        f'First missing path: {missing_paths[0]}'
    )

train_mask = teacher_meta['fold'].to_numpy() != CFG.fold
valid_mask = teacher_meta['fold'].to_numpy() == CFG.fold

train_frame = teacher_meta.loc[train_mask].reset_index(drop=True)
valid_frame = teacher_meta.loc[valid_mask].reset_index(drop=True)
train_labels_fold = teacher_labels[train_mask]
valid_labels_fold = teacher_labels[valid_mask]
train_teacher_probs = teacher_probs[train_mask]
valid_teacher_probs = teacher_probs[valid_mask]

if CFG.min_teacher_confidence_train > 0:
    keep = train_frame['teacher_confidence'].to_numpy() >= CFG.min_teacher_confidence_train
    train_frame = train_frame.loc[keep].reset_index(drop=True)
    train_labels_fold = train_labels_fold[keep]
    train_teacher_probs = train_teacher_probs[keep]

if CFG.max_train_rows is not None:
    keep = min(CFG.max_train_rows, len(train_frame))
    selected = train_frame.sample(keep, random_state=CFG.random_seed).index.to_numpy()
    train_frame = train_frame.loc[selected].reset_index(drop=True)
    train_labels_fold = train_labels_fold[selected]
    train_teacher_probs = train_teacher_probs[selected]
if CFG.max_valid_rows is not None:
    keep = min(CFG.max_valid_rows, len(valid_frame))
    selected = valid_frame.sample(keep, random_state=CFG.random_seed).index.to_numpy()
    valid_frame = valid_frame.loc[selected].reset_index(drop=True)
    valid_labels_fold = valid_labels_fold[selected]
    valid_teacher_probs = valid_teacher_probs[selected]

fold_overview = {
    'fold': CFG.fold,
    'train_rows': int(len(train_frame)),
    'valid_rows': int(len(valid_frame)),
    'train_files': int(train_frame['filename'].nunique()),
    'valid_files': int(valid_frame['filename'].nunique()),
    'mean_train_teacher_confidence': float(train_frame['teacher_confidence'].mean()),
    'mean_valid_teacher_confidence': float(valid_frame['teacher_confidence'].mean()),
}
print(json.dumps(fold_overview, indent=2))


{
  "fold": 0,
  "train_rows": 540,
  "valid_rows": 168,
  "train_files": 45,
  "valid_files": 14,
  "mean_train_teacher_confidence": 0.9708983302116394,
  "mean_valid_teacher_confidence": 0.9788637757301331
}


In [17]:
def read_audio_region(path: str, clip_start_frame: int, clip_end_frame: int, sample_frames: int, training: bool) -> np.ndarray:
    with sf.SoundFile(path) as snd:
        total_frames = snd.frames
        region_start = max(int(clip_start_frame), 0)
        region_end = total_frames if int(clip_end_frame) <= 0 else min(int(clip_end_frame), total_frames)
        region_frames = max(region_end - region_start, 0)
        if region_frames <= 0:
            return np.zeros(sample_frames, dtype=np.float32)
        if region_frames >= sample_frames:
            if training:
                offset = np.random.randint(region_frames - sample_frames + 1)
            else:
                offset = 0
            snd.seek(region_start + offset)
            wave = snd.read(frames=sample_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            if wave.shape[0] == sample_frames:
                return wave
        else:
            snd.seek(region_start)
            wave = snd.read(frames=region_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)

    actual_frames = int(wave.shape[0])
    if actual_frames >= sample_frames:
        return wave[:sample_frames]
    padded = np.zeros(sample_frames, dtype=np.float32)
    pad_start = np.random.randint(sample_frames - actual_frames + 1) if training else 0
    padded[pad_start: pad_start + actual_frames] = wave
    return padded


class DistillSoundscapeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, label_arr: np.ndarray, teacher_probs_arr: np.ndarray, training: bool):
        self.frame = frame.reset_index(drop=True)
        self.labels = label_arr.astype(np.float32)
        self.teacher_probs = teacher_probs_arr.astype(np.float32)
        self.training = training
        self.sample_frames = CFG.sample_rate * CFG.segment_seconds
        self.teacher_weights = np.power(
            np.clip(self.frame['teacher_confidence'].to_numpy(dtype=np.float32), 1e-3, 1.0),
            CFG.teacher_weight_power,
        ).astype(np.float32)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, tp.Any]:
        row = self.frame.iloc[idx]
        wave = read_audio_region(
            path=str(row['file_path']),
            clip_start_frame=int(row['clip_start_frame']),
            clip_end_frame=int(row['clip_end_frame']),
            sample_frames=self.sample_frames,
            training=self.training,
        )
        return {
            'wave': wave,
            'label': self.labels[idx],
            'teacher_prob': self.teacher_probs[idx],
            'teacher_weight': self.teacher_weights[idx],
            'index': idx,
        }


class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            pad_mode='reflect',
            norm='slaney',
            mel_scale='htk',
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        if wave.ndim == 1:
            wave = wave.unsqueeze(0)
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        flat = mel.flatten(start_dim=1)
        min_val = flat.min(dim=1).values[:, None, None, None]
        max_val = flat.max(dim=1).values[:, None, None, None]
        mel = (mel - min_val) / (max_val - min_val + 1e-7)
        return mel


def make_loader(dataset: Dataset, training: bool) -> DataLoader:
    return DataLoader(
        dataset=dataset,
        batch_size=CFG.batch_size,
        shuffle=training,
        drop_last=training and len(dataset) >= CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == 'cuda'),
    )


def compute_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int]:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    scored_classes = int(scored_mask.sum())
    skipped_no_positive = int((~positive_mask).sum())
    skipped_no_negative = int((~negative_mask).sum())
    if scored_classes == 0:
        return {
            'macro_auc': np.nan,
            'scored_classes': scored_classes,
            'skipped_no_positive': skipped_no_positive,
            'skipped_no_negative': skipped_no_negative,
        }
    macro_auc = roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro')
    return {
        'macro_auc': float(macro_auc),
        'scored_classes': scored_classes,
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def evaluate_predictions(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int]:
    overall = compute_macro_auc(y_true, y_score)
    return {
        **overall,
        'soundscape_macro_auc': overall['macro_auc'],
        'soundscape_scored_classes': overall['scored_classes'],
        'soundscape_skipped_no_positive': overall['skipped_no_positive'],
        'soundscape_skipped_no_negative': overall['skipped_no_negative'],
    }


def build_model() -> nn.Module:
    return timm.create_model(
        CFG.model_name,
        pretrained=CFG.pretrained,
        in_chans=1,
        num_classes=N_CLASSES,
        drop_path_rate=CFG.drop_path_rate,
    )


def load_pretrained_state(model: nn.Module, checkpoint_path: Path) -> None:
    payload = torch.load(checkpoint_path, map_location='cpu')
    state_dict = payload['model_state_dict'] if 'model_state_dict' in payload else payload
    model.load_state_dict(state_dict, strict=True)


def load_resume_payload(output_dir: Path) -> dict[str, tp.Any] | None:
    last_path = output_dir / 'last_model.pt'
    if not last_path.exists():
        return None
    return torch.load(last_path, map_location='cpu')


In [18]:
def train_one_fold(
    train_frame: pd.DataFrame,
    valid_frame: pd.DataFrame,
    train_labels: np.ndarray,
    valid_labels: np.ndarray,
    train_teacher_probs: np.ndarray,
    valid_teacher_probs: np.ndarray,
    output_dir: Path,
) -> tuple[pd.DataFrame, dict[str, tp.Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = DistillSoundscapeDataset(train_frame, train_labels, train_teacher_probs, training=True)
    valid_dataset = DistillSoundscapeDataset(valid_frame, valid_labels, valid_teacher_probs, training=False)
    train_loader = make_loader(train_dataset, training=True)
    valid_loader = make_loader(valid_dataset, training=False)

    mel_transform = LogMelSpectrogramTransform().to(device).eval()
    model = build_model().to(device)
    init_ckpt = EXP011_DIR / f'fold_{CFG.fold:02d}' / 'best_model.pt'
    if init_ckpt.exists():
        load_pretrained_state(model, init_ckpt)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = OneCycleLR(
        optimizer=optimizer,
        max_lr=CFG.learning_rate,
        epochs=CFG.epochs,
        steps_per_epoch=max(1, len(train_loader)),
        pct_start=max(1, CFG.warmup_epochs) / max(1, CFG.epochs),
        div_factor=25,
        final_div_factor=4.0,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    history: list[dict[str, tp.Any]] = []
    start_epoch = 1
    best_metric = -float('inf')
    resume_mode = 'init_from_exp011_fold'

    if RESUME_TRAINING:
        payload = load_resume_payload(output_dir)
        if payload is not None:
            model.load_state_dict(payload['model_state_dict'])
            optimizer.load_state_dict(payload['optimizer_state_dict'])
            scheduler.load_state_dict(payload['scheduler_state_dict'])
            scaler_state = payload.get('scaler_state_dict')
            if scaler_state is not None and amp_enabled:
                scaler.load_state_dict(scaler_state)
            history = payload.get('history', [])
            start_epoch = int(payload.get('epoch', 0)) + 1
            best_metric = float(payload.get('best_metric', -float('inf')))
            resume_mode = 'resume_exp027b'

    for epoch in range(start_epoch, CFG.epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_sup_sum = 0.0
        train_distill_sum = 0.0

        for batch in tqdm(train_loader, desc=f'epoch {epoch} train', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            label = batch['label'].to(device, dtype=torch.float32)
            teacher_prob = batch['teacher_prob'].to(device, dtype=torch.float32)
            teacher_weight = batch['teacher_weight'].to(device, dtype=torch.float32)

            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                image = mel_transform(wave)
            with autocast_context():
                logits = model(image)
                supervised_loss = F.binary_cross_entropy_with_logits(logits, label)
                distill_vec = F.binary_cross_entropy_with_logits(
                    logits / CFG.distill_temperature,
                    teacher_prob,
                    reduction='none',
                ).mean(dim=1)
                distill_loss = (distill_vec * teacher_weight).mean() * (CFG.distill_temperature ** 2)
                loss = supervised_loss + CFG.distill_weight * distill_loss
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            scheduler.step()

            train_loss_sum += float(loss.item())
            train_sup_sum += float(supervised_loss.item())
            train_distill_sum += float(distill_loss.item())
            del wave, label, teacher_prob, teacher_weight, image, logits, supervised_loss, distill_loss, loss

        train_loss = train_loss_sum / max(1, len(train_loader))
        train_sup = train_sup_sum / max(1, len(train_loader))
        train_distill = train_distill_sum / max(1, len(train_loader))

        model.eval()
        valid_loss_sum = 0.0
        logits_parts: list[np.ndarray] = []
        label_parts: list[np.ndarray] = []
        index_parts: list[np.ndarray] = []
        teacher_conf_parts: list[np.ndarray] = []

        for batch in tqdm(valid_loader, desc=f'epoch {epoch} valid', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            label = batch['label'].to(device, dtype=torch.float32)
            teacher_prob = batch['teacher_prob'].to(device, dtype=torch.float32)
            teacher_weight = batch['teacher_weight'].to(device, dtype=torch.float32)
            with torch.no_grad():
                image = mel_transform(wave)
                with autocast_context():
                    logits = model(image)
                    supervised_loss = F.binary_cross_entropy_with_logits(logits, label)
                    distill_vec = F.binary_cross_entropy_with_logits(
                        logits / CFG.distill_temperature,
                        teacher_prob,
                        reduction='none',
                    ).mean(dim=1)
                    distill_loss = (distill_vec * teacher_weight).mean() * (CFG.distill_temperature ** 2)
                    loss = supervised_loss + CFG.distill_weight * distill_loss
            valid_loss_sum += float(loss.item())
            logits_parts.append(logits.detach().float().cpu().numpy())
            label_parts.append(label.detach().float().cpu().numpy())
            index_parts.append(batch['index'].detach().cpu().numpy())
            teacher_conf_parts.append(teacher_weight.detach().float().cpu().numpy())
            del wave, label, teacher_prob, teacher_weight, image, logits, supervised_loss, distill_loss, loss

        valid_loss = valid_loss_sum / max(1, len(valid_loader))
        logits_all = np.concatenate(logits_parts, axis=0)
        labels_all = np.concatenate(label_parts, axis=0)
        index_all = np.concatenate(index_parts, axis=0)
        order = np.argsort(index_all)
        logits_all = logits_all[order]
        labels_all = labels_all[order]
        probs_all = (1.0 / (1.0 + np.exp(-logits_all))).astype(np.float32)
        valid_metrics = evaluate_predictions(labels_all, probs_all)

        selected_metric = valid_metrics[CFG.selection_metric]
        if np.isnan(selected_metric):
            selected_metric = valid_metrics['macro_auc']

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_supervised_loss': train_sup,
            'train_distill_loss': train_distill,
            'macro_auc': valid_metrics['macro_auc'],
            'scored_classes': valid_metrics['scored_classes'],
            'skipped_no_positive': valid_metrics['skipped_no_positive'],
            'skipped_no_negative': valid_metrics['skipped_no_negative'],
            'soundscape_macro_auc': valid_metrics['soundscape_macro_auc'],
            'soundscape_scored_classes': valid_metrics['soundscape_scored_classes'],
            'soundscape_skipped_no_positive': valid_metrics['soundscape_skipped_no_positive'],
            'soundscape_skipped_no_negative': valid_metrics['soundscape_skipped_no_negative'],
            'valid_loss': valid_loss,
            'learning_rate': float(scheduler.get_last_lr()[0]),
            'selection_metric': float(selected_metric),
            'distill_weight': float(CFG.distill_weight),
            'mean_train_teacher_confidence': float(train_frame['teacher_confidence'].mean()),
        }
        history.append(row)
        history_df = pd.DataFrame(history)
        history_df.to_csv(output_dir / 'history.csv', index=False)

        payload = {
            'epoch': epoch,
            'best_metric': max(best_metric, float(selected_metric)),
            'history': history,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if amp_enabled else None,
            'cfg': asdict(CFG),
            'classes': CLASSES,
            'resume_mode': resume_mode,
        }
        if CFG.save_every_epoch:
            torch.save(payload, output_dir / 'last_model.pt')

        if float(selected_metric) > best_metric:
            best_metric = float(selected_metric)
            torch.save(payload, output_dir / 'best_model.pt')
            np.savez_compressed(
                output_dir / 'best_valid_outputs.npz',
                logits=logits_all.astype(np.float32),
                probs=probs_all.astype(np.float32),
                labels=labels_all.astype(np.float32),
            )
            valid_frame.reset_index(drop=True).to_csv(output_dir / 'best_valid_meta.csv', index=False)

        print(row)

    history_df = pd.DataFrame(history)
    best_row = history_df.loc[history_df['selection_metric'].idxmax()].to_dict()
    snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'fold': CFG.fold,
        'best_epoch': int(best_row['epoch']),
        'best_selection_metric': float(best_row['selection_metric']),
        'best_macro_auc': float(best_row['macro_auc']),
        'best_soundscape_macro_auc': float(best_row['soundscape_macro_auc']),
        'scored_classes': int(best_row['scored_classes']),
        'soundscape_scored_classes': int(best_row['soundscape_scored_classes']),
        'best_valid_loss': float(best_row['valid_loss']),
        'train_rows': int(len(train_frame)),
        'valid_rows': int(len(valid_frame)),
        'resume_mode': resume_mode,
        'teacher_dir': str(TEACHER_DIR),
        'output_dir': str(output_dir),
        'model_name': CFG.model_name,
        'distill_weight': float(CFG.distill_weight),
        'device': str(device),
    }
    save_json(snapshot, output_dir / 'result_snapshot.json')
    return history_df, snapshot


if RUN_TRAINING:
    history_df, snapshot = train_one_fold(
        train_frame,
        valid_frame,
        train_labels_fold,
        valid_labels_fold,
        train_teacher_probs,
        valid_teacher_probs,
        OUTPUT_DIR,
    )
else:
    history_df = None
    snapshot = None


epoch 1 train:   0%|          | 0/33 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 1 valid:   0%|          | 0/11 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


{'epoch': 1, 'train_loss': 0.5550886528058485, 'train_supervised_loss': 0.08721671840458205, 'train_distill_loss': 1.3367769682046138, 'macro_auc': 0.9347742937673617, 'scored_classes': 43, 'skipped_no_positive': 191, 'skipped_no_negative': 0, 'soundscape_macro_auc': 0.9347742937673617, 'soundscape_scored_classes': 43, 'soundscape_skipped_no_positive': 191, 'soundscape_skipped_no_negative': 0, 'valid_loss': 0.4058394134044647, 'learning_rate': 0.00010631971954266869, 'selection_metric': 0.9347742937673617, 'distill_weight': 0.35, 'mean_train_teacher_confidence': 0.9708983302116394}


...

epoch 8 train:   0%|          | 0/33 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 8 valid:   0%|          | 0/11 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


{'epoch': 8, 'train_loss': 0.2921897362578999, 'train_supervised_loss': 0.0815477529258439, 'train_distill_loss': 0.6018342411879337, 'macro_auc': 0.9596433424596299, 'scored_classes': 43, 'skipped_no_positive': 191, 'skipped_no_negative': 0, 'soundscape_macro_auc': 0.9596433424596299, 'soundscape_scored_classes': 43, 'soundscape_skipped_no_positive': 191, 'soundscape_skipped_no_negative': 0, 'valid_loss': 0.30555095997723664, 'learning_rate': 2.012461360286368e-06, 'selection_metric': 0.9596433424596299, 'distill_weight': 0.35, 'mean_train_teacher_confidence': 0.9708983302116394}


In [19]:
if (OUTPUT_DIR / 'result_snapshot.json').exists():
    snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))
    if (OUTPUT_DIR / 'history.csv').exists():
        display(pd.read_csv(OUTPUT_DIR / 'history.csv'))
else:
    print('No training artifacts yet. Set RUN_TRAINING = True to start the experiment.')


Snapshot:
{
  "experiment_id": "exp_027b",
  "experiment_name": "hgnetv2_soundscape_distill_from_exp015d",
  "fold": 0,
  "best_epoch": 5,
  "best_selection_metric": 0.9610550772798211,
  "best_macro_auc": 0.9610550772798211,
  "best_soundscape_macro_auc": 0.9610550772798211,
  "scored_classes": 43,
  "soundscape_scored_classes": 43,
  "best_valid_loss": 0.30462846159935,
  "train_rows": 540,
  "valid_rows": 168,
  "resume_mode": "init_from_exp011_fold",
  "teacher_dir": "/Users/yaroslav/Downloads/exp_027a_exp015d_teacher_cache",
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_027b_hgnetv2_soundscape_distill_from_exp015d/fold_00",
  "model_name": "hgnetv2_b0.ssld_stage2_ft_in1k",
  "distill_weight": 0.35,
  "device": "mps"
}


,epoch,train_loss,train_supervised_loss,train_distill_loss,macro_auc,scored_classes,skipped_no_positive,skipped_no_negative,soundscape_macro_auc,soundscape_scored_classes,soundscape_skipped_no_positive,soundscape_skipped_no_negative,valid_loss,learning_rate,selection_metric,distill_weight,mean_train_teacher_confidence
0,1,0.555089,0.087217,1.336777,0.934774,43,191,0,0.934774,43,191,0,0.405839,0.000106,0.934774,0.35,0.970898
1,2,0.339350,0.099153,0.686278,0.944684,43,191,0,0.944684,43,191,0,0.316471,0.000200,0.944684,0.35,0.970898
2,3,0.309282,0.091521,0.622175,0.958672,43,191,0,0.958672,43,191,0,0.307393,0.000186,0.958672,0.35,0.970898
3,4,0.300710,0.087043,0.610477,0.959187,43,191,0,0.959187,43,191,0,0.305196,0.000149,0.959187,0.35,0.970898
4,5,0.296061,0.084238,0.605209,0.961055,43,191,0,0.961055,43,191,0,0.304628,0.000099,0.961055,0.35,0.970898
5,6,0.294070,0.082453,0.604619,0.959578,43,191,0,0.959578,43,191,0,0.305060,0.000050,0.959578,0.35,0.970898
6,7,0.292010,0.082046,0.599897,0.960028,43,191,0,0.960028,43,191,0,0.305582,0.000014,0.960028,0.35,0.970898
7,8,0.292190,0.081548,0.601834,0.959643,43,191,0,0.959643,43,191,0,0.305551,0.000002,0.959643,0.35,0.970898
